# Macaron drawing style test\nDraws a set of dishes in the chosen style so they can be checked. Only a contact sheet is saved.

In [ ]:
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
!python -c "import diffusers" 2>/dev/null || pip -q install --no-deps diffusers 2>&1 | tail -2

In [ ]:
import torch, time
from PIL import Image, ImageDraw, ImageFont
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from rembg import remove, new_session
pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe = pipe.to("cuda"); pipe.vae.enable_slicing()
CUT = new_session("isnet-general-use")
NEG = "photo, photograph, photorealistic, realistic, text, watermark, logo, hands, person, chopsticks, table, tablecloth, blurry, lowres, deformed, messy, cluttered, dark background"
def gen(p, seed):
    g = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=p, negative_prompt=NEG, width=1024, height=1024, num_inference_steps=28, guidance_scale=6.5, generator=g).images[0]
def cut(img):
    o = remove(img, session=CUT).convert("RGBA"); bb = o.getbbox()
    if bb: o = o.crop(bb)
    o.thumbnail((300, 300), Image.LANCZOS); return o

In [ ]:
DISHES = {
 "Nasi lemak biasa": "Malaysian nasi lemak on a white plate: a dome of white coconut rice on a piece of banana leaf, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices, and half a hard-boiled egg",
 "Chicken rice, roasted": "Malaysian roasted chicken rice: neat slices of roasted chicken thigh with glossy golden-brown skin laid over cucumber slices, a separate dome of fragrant rice, a small dish of red chilli sauce, no whole chicken",
 "Chicken rice, steamed": "Malaysian Hainanese steamed chicken rice: neat slices of pale poached chicken with smooth skin over cucumber slices, drizzled with light soy sauce and sesame oil, a dome of rice, a small dish of chilli sauce",
 "Curry laksa": "Malaysian curry laksa in a white bowl: orange coconut curry soup, yellow noodles, tofu puffs, prawns, half a boiled egg, bean sprouts, mint leaves",
 "Prawn mee": "Penang prawn mee in a bowl: red-orange prawn broth, yellow noodles and rice vermicelli, whole prawns, pork slices, half a boiled egg, water spinach, fried shallots",
 "Wantan mee, dry": "Malaysian dry wantan mee on a plate: thin springy egg noodles tossed in dark soy sauce, slices of red-edged char siu barbecued pork, choy sum greens, a small bowl of wonton soup on the side, no fried egg",
 "Char kuey teow": "Penang char kuey teow on a plate: flat rice noodles stir-fried brown with dark soy sauce, prawns, cockles, bean sprouts, chives and bits of scrambled egg, no fried egg",
 "Fried wantan": "golden deep-fried wontons on a small plate: round puffy pouches with crinkly frilled crispy edges, not pointed, with a small dish of sweet chilli sauce",
 "Kaya toast set": "Malaysian kaya toast set: thin toasted bread sandwiches filled with green kaya jam and a thick slice of cold butter, a small bowl with two runny soft-boiled eggs with soy sauce, a cup of black kopi",
 "Roti canai kosong": "Malaysian roti canai: two round flaky golden flatbreads loosely folded on a plate with a small bowl of yellow dhal curry",
 "Mee goreng mamak": "Malaysian mee goreng mamak: yellow noodles fried in a red-orange spicy sauce with tofu, potato, bean sprouts, egg and a lime wedge on a plate",
 "Nasi goreng kampung": "Malaysian nasi goreng kampung: fried rice with anchovies and water spinach, a sunny-side-up fried egg on top, on a plate",
 "Satay ayam": "Malaysian chicken satay: six grilled chicken skewers with caramelised edges, a bowl of peanut sauce, cucumber and red onion pieces, rice cakes",
 "Banana leaf rice": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam, and fish curry gravy",
 "Hokkien mee": "KL Hokkien mee: thick yellow noodles braised in glossy dark soy sauce with pork slices, prawns, cabbage and crispy pork lard bits on a plate",
 "Yong tau foo": "Malaysian yong tau foo: stuffed tofu, fish balls, stuffed red chilli, stuffed bitter gourd and greens in a bowl of clear soup",
 "Bak kut teh": "Malaysian bak kut teh: a clay pot of dark herbal soup with pork ribs, tofu puffs, mushrooms and lettuce",
 "Braised beef noodle soup": "Taiwanese braised beef noodle soup in a bowl: dark brown broth, thick wheat noodles, chunks of braised beef shank, bok choy, chopped spring onion",
 "Pho bo": "Vietnamese beef pho in a bowl: clear broth, flat rice noodles, thin slices of beef, basil, bean sprouts, a lime wedge",
 "Teh tarik": "a glass mug of Malaysian teh tarik milk tea, light brown with a thick frothy foam top",
 "Curry puff": "two Malaysian curry puffs: golden half-moon pastries with crimped twisted edges",
 "Butter croissant": "one golden flaky butter croissant",
 "Cendol": "a bowl of Malaysian cendol: shaved ice with green rice-flour jelly strands, red beans, coconut milk and dark palm sugar syrup",
 "Ice cream, waffle cone": "two scoops of ice cream, strawberry and vanilla, in a crisp waffle cone"
}
STYLE = "a soft digital painting illustration of {d}, smooth painterly shading, gentle highlights, warm soft light, one cohesive illustration, cookbook illustration, three-quarter view, centered, plain white background"
res = {}
for d, desc in DISHES.items():
    t = time.time(); res[d] = cut(gen(STYLE.format(d=desc), 17)); print(d, round(time.time() - t), "s")

In [ ]:
C = 250; W = 6; names = list(DISHES); H = (len(names) + W - 1) // W
sheet = Image.new("RGB", (W * C, H * (C + 30) + 10), "#fff0f4"); dr = ImageDraw.Draw(sheet)
try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 15)
except Exception: f = ImageFont.load_default()
for k, d in enumerate(names):
    x, y = (k % W) * C, (k // W) * (C + 30) + 8
    im = res[d].copy(); im.thumbnail((C - 20, C - 20)); sheet.paste(im, (x + (C - im.width) // 2, y + (C - 10 - im.height) // 2), im)
    dr.text((x + 10, y + C - 4), d, fill="#7a2e4d", font=f)
sheet.save("/kaggle/working/contact_styles.jpg", quality=88); print("sheet saved", sheet.size)